In [0]:
%sql
select * from silicon.lp_master;

In [0]:
from pathlib import Path

repo_root = Path.cwd().parent     
result_file = repo_root / "lammps" / "Si" / "results" / "bulk_modulus" / "E_vs_V.txt"


In [0]:
E_vs_V_df = (
    spark.read
    .option("delimiter", " ")
    .option("header", "true")
    .csv(str(result_file))
)

In [0]:
E_vs_V_df.show()

In [0]:
E_vs_V_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silicon.lp_vs_energy_vs_vol")

In [0]:
%sql

select * from silicon.total_energy_vs_volume

In [0]:

# Compute Bulk modulus

from pyspark.sql import functions as F

# Read Delta table
df = spark.read.table("silicon.lp_vs_energy_vs_vol")

# Keep only required columns
df = (
    df.select(
        F.col("Volume").cast("double"),
        F.col("Total_energy").cast("double")
    )
    .orderBy("Total_energy")
)

display(df)

In [0]:
pdf = df.toPandas()

V = pdf["Volume"].to_numpy()
E = pdf["Total_energy"].to_numpy()

import numpy as np
from scipy.optimize import curve_fit

def birch_murnaghan(V, E0, V0, B0, B0_prime):

    eta = (V0 / V)**(2.0/3.0)

    return E0 + (9.0 * V0 * B0 / 16.0) * (
        (eta - 1.0)**3 * B0_prime +
        (eta - 1.0)**2 * (6.0 - 4.0 * eta)
    )
# Initial guesses
E0_guess = E.min()
V0_guess = V[np.argmin(E)]
B0_guess = 0.6      # ≈99 GPa
B0_prime_guess = 4.0

params, _ = curve_fit(
    birch_murnaghan,
    V,
    E,
    p0=[E0_guess, V0_guess, 0.62, 4.0]
)
E0, V0, B0, B0_prime = params

bulk_modulus = B0 * 160.21766208

print(f"Equilibrium volume : {V0:.4f} Å³")
print(f"Minimum energy     : {E0:.6f} eV")
print(f"Bulk modulus       : {bulk_modulus:.2f} GPa")
print(f"B0'                : {B0_prime:.3f}")